# 02.2 — Chart Auto-Encoder Validity Test

**Execution of a pre-registered analysis.** The architecture, training protocol, three
gates, and their thresholds were committed in `02.2-PREREGISTRATION.md` (ratified before
any fit ran, CAE-01) — no threshold, constant, or metric definition here was chosen after
seeing a result.

**The three gates, applied as a strict-less-than conjunction, no MARGINAL tier:**

| Gate | Measures | Threshold |
|---|---|---|
| T1 `distortion` | geodesic distortion of the initial encoder's global embedding | `< 0.15` |
| T2 `rcycle_ratio` | chart-transition cycle residual, normalised by a matched base | `< 2.0` |
| T3 `recon_margin` | held-out reconstruction vs. matched-capacity controls (worst ratio) | `< 0.90` |

PASS requires all three. Every comparison is strict less-than; a value exactly at a
threshold does not clear it. There is no MARGINAL tier — every non-PASS outcome routes to
the same halt-for-user-decision consequence.

**Binding prohibitions for this notebook specifically.** It trains nothing: all eight fits
(three CAE seeds, one ReLU control, two plain-AE controls, two MDS-decoder baselines) were
already trained by plan 02.2-05 and are reloaded here, never refit. It writes nothing into
`notebooks/.cache/`: the sealed verdict artifact it regression-checks against
(`cae_verdict_43cf438bc944c509.json`) is strictly read-only to it. It revises no threshold.
And it reproduces a recorded **FAIL** — it is not searching for a PASS, and a FAIL is not
an error to work around.


## §1. Environment and provenance

Package versions participate in the sealed artifact's own recorded `versions` block, so a
live version drift would make §10's floating-point regression check untrustworthy. This
cell asserts the four live versions match before anything downstream runs, and pins
`FIT_KEY`/`SUBSAMPLE_STEM`, the two frozen identifiers naming which fit and which subsample
every subsequent cell reloads.


In [1]:
import gc
import json
import resource
import subprocess
import sys
from pathlib import Path

# Make the notebook-local pu_manifold package importable exactly as notebook 02 does
# (plain relative import, never installed, never imported from src/effdim/).
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
assert NOTEBOOK_DIR.name == "notebooks", (
    f"NOTEBOOK_DIR={NOTEBOOK_DIR!r} does not end in 'notebooks'. Every downstream "
    "`sys.path`/`pu_manifold` import resolves relative to the *kernel's* working "
    "directory, not this file's location on disk -- a kernel launched from anywhere "
    "else either fails to import pu_manifold or silently resolves a different package "
    "of the same name. Launch this kernel with notebooks/ as the working directory."
)

import numpy as np
import scipy
import torch

from pu_manifold import cache
from pu_manifold import cae as cae_mod
from pu_manifold import geometry_probes as gp

# --- frozen identifiers (established before this plan; not re-derived here) --------------
FIT_KEY = "43cf438bc944c509"
SUBSAMPLE_STEM = "subsample_20260729_a79b3460b838fd0a"

# Load the sealed artifact read-only: every metric this notebook recomputes is checked
# against it, but nothing here ever writes back to it.
PUBLISHED = json.loads(cache.cache_path(f"cae_verdict_{FIT_KEY}", "json").read_text())

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("=== Reproducibility header (02.2) ===")
print(f"python         = {sys.version.split()[0]}")
print(f"numpy          = {np.__version__}")
print(f"scipy          = {scipy.__version__}")
print(f"torch          = {torch.__version__}")
print(f"git commit SHA = {git_sha}")
print(f"cwd            = {NOTEBOOK_DIR}")
print(f"FIT_KEY        = {FIT_KEY}")
print(f"SUBSAMPLE_STEM = {SUBSAMPLE_STEM}")

_live_versions = {
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "torch": torch.__version__,
    "python": sys.version.split()[0],
}
for _k, _v in _live_versions.items():
    assert _v == PUBLISHED["versions"][_k], (
        f"Live {_k} version {_v!r} does not match the sealed artifact's recorded "
        f"{PUBLISHED['versions'][_k]!r}. §10's regression check compares floating-point "
        "results against a run made under those exact versions and cannot be trusted "
        "across a version drift -- halting before computing anything."
    )
print("\nAll four live versions match PUBLISHED['versions'] exactly.")


=== Reproducibility header (02.2) ===
python         = 3.14.6
numpy          = 2.5.1
scipy          = 1.18.0
torch          = 2.13.0+cpu
git commit SHA = c90eea9
cwd            = /home/akagi/Documents/Projects/EffDim/notebooks
FIT_KEY        = 43cf438bc944c509
SUBSAMPLE_STEM = subsample_20260729_a79b3460b838fd0a

All four live versions match PUBLISHED['versions'] exactly.


## §2. Pre-registered constants

Every constant below is fixed in `02.2-PREREGISTRATION.md` Section 4, copied here verbatim
as named module-level values -- never inlined as a magic number at a call site. The asserts
following the declarations cross-check the subset the sealed artifact independently
records, so a drift between this notebook's constants and the ones the sealed run actually
used is caught here rather than assumed.


In [2]:
# --- architecture -------------------------------------------------------------------------
D_CHART = 20            # chart-space dimension, each chart maps into (0,1)^D_CHART
L_EMBED = 40             # global embedding dimension (initial encoder's output space)
N_CHARTS_INIT = 16       # initial (over-specified) chart count, before a-posteriori pruning
HIDDEN_WIDTH = 250       # hidden width, three layers, every MLP in the architecture
ACTIVATION = "silu"      # C2-smooth activation (ReLU only ever appears as the CAE-06 control)

# --- optimisation -------------------------------------------------------------------------
LIP_WEIGHT = 1e-2                # eq. 4 Lipschitz penalty weight
LIP_EVERY_N_STEPS = 1            # penalty applied every step (pre-registered default)
WEIGHT_DECAY = 1e-4
ADAM_LR = 3e-4
BATCH = 64
MAX_EPOCHS = 40
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 1e-4
WALLCLOCK_CEILING_S = 7200
FPS_PRETRAIN_EPOCHS = 5           # eq. 5 FPS-seeded per-chart pre-training epochs

# --- gating thresholds (Section 5's verdict rule; strict less-than, no MARGINAL tier) ------
THRESH_DISTORTION_MAX = 0.15
THRESH_RCYCLE_RATIO_MAX = 2.0
THRESH_RECON_MARGIN = 0.10        # gate threshold is 1 - THRESH_RECON_MARGIN, see §8/§10

# --- chart survival / overlap-pair selection -----------------------------------------------
PRUNE_TOL = 1e-2
OVERLAP_P_MIN = 0.05
OVERLAP_MIN_POINTS = 200

# --- splits and seeds -----------------------------------------------------------------------
HOLDOUT_FRACTION = 0.2
SPLIT_SEED = 20260803
SEEDS = (20260803, 20260804, 20260805)
MAIN_SEED = 20260803
UNFAITHFUL_SAMPLES = 1000
UNFAITHFUL_SEED = 20260806
PAIR_SEED = 20260731
PAIR_COUNT = 200000

# --- control ladders (CAE-03 matched-capacity baselines) ------------------------------------
MDS_BASELINE_P = (8, 20)
PLAIN_AE_LATENTS = (20, 40)

# --- cross-check the subset the sealed artifact independently records ----------------------
assert THRESH_DISTORTION_MAX == PUBLISHED["thresholds"]["distortion"], (
    f"THRESH_DISTORTION_MAX={THRESH_DISTORTION_MAX} vs sealed "
    f"{PUBLISHED['thresholds']['distortion']}"
)
assert THRESH_RCYCLE_RATIO_MAX == PUBLISHED["thresholds"]["rcycle_ratio"], (
    f"THRESH_RCYCLE_RATIO_MAX={THRESH_RCYCLE_RATIO_MAX} vs sealed "
    f"{PUBLISHED['thresholds']['rcycle_ratio']}"
)
assert 1.0 - THRESH_RECON_MARGIN == PUBLISHED["thresholds"]["recon_margin"], (
    f"1-THRESH_RECON_MARGIN={1.0 - THRESH_RECON_MARGIN} vs sealed "
    f"{PUBLISHED['thresholds']['recon_margin']}"
)
assert list(SEEDS) == PUBLISHED["seeds"]["all_seeds"], (
    f"SEEDS={list(SEEDS)} vs sealed {PUBLISHED['seeds']['all_seeds']}"
)
assert MAIN_SEED == PUBLISHED["seeds"]["main_seed"], (
    f"MAIN_SEED={MAIN_SEED} vs sealed {PUBLISHED['seeds']['main_seed']}"
)
assert HOLDOUT_FRACTION == PUBLISHED["holdout"]["holdout_fraction"], (
    f"HOLDOUT_FRACTION={HOLDOUT_FRACTION} vs sealed {PUBLISHED['holdout']['holdout_fraction']}"
)
assert SPLIT_SEED == PUBLISHED["holdout"]["split_seed"], (
    f"SPLIT_SEED={SPLIT_SEED} vs sealed {PUBLISHED['holdout']['split_seed']}"
)
assert OVERLAP_MIN_POINTS == PUBLISHED["holdout"]["t2_min_points_required"], (
    f"OVERLAP_MIN_POINTS={OVERLAP_MIN_POINTS} vs sealed "
    f"{PUBLISHED['holdout']['t2_min_points_required']}"
)
assert PRUNE_TOL == PUBLISHED["chart_count_surviving"]["prune_tol"], (
    f"PRUNE_TOL={PRUNE_TOL} vs sealed {PUBLISHED['chart_count_surviving']['prune_tol']}"
)

print("=== §2: pre-registered constants, cross-checked against the sealed artifact ===")
print(f"T1 distortion   < {THRESH_DISTORTION_MAX}")
print(f"T2 rcycle_ratio < {THRESH_RCYCLE_RATIO_MAX}")
print(f"T3 recon_margin < {1.0 - THRESH_RECON_MARGIN}  (= 1 - THRESH_RECON_MARGIN)")
print(f"\nVerdict rule: {cae_mod.VERDICT_RULE}")


=== §2: pre-registered constants, cross-checked against the sealed artifact ===
T1 distortion   < 0.15
T2 rcycle_ratio < 2.0
T3 recon_margin < 0.9  (= 1 - THRESH_RECON_MARGIN)

Verdict rule: PASS requires all three gates to hold. Every comparison is strict less-than -- a value exactly at a threshold does not clear it. There is no MARGINAL tier: every non-PASS outcome routes to the same halt-for-user-decision consequence, so a middle tier would carry no distinct consequence.


## §3. The eight cached fits, and what training them cost

One `ChartAutoEncoder`: an initial encoder into `R^40`, sixteen over-specified chart
encoders mapping into `(0,1)^20`, a per-chart decoder each, one shared embedding decoder,
and a softmax chart predictor (partition of unity). Trained against the paper's own loss
(eq. 3, reconstruction plus a cross-entropy term against a detached softmax target) plus a
Lipschitz penalty (eq. 4) on chart-encoder spectral norms, preceded by five epochs of
FPS-seeded per-chart pre-training (eq. 5) -- without which the second chart never activates
(the dead-chart failure mode the paper warns about). Optimised with AdamW at lr=3e-4, batch
64, up to 40 epochs, stopped by whichever of three limits fires first: the epoch cap, a
relative-plateau early-stop at patience 5, or a 7200s wallclock ceiling.

Eight such fits exist: three CAE seeds (the main and two replicate seeds), one ReLU-
activation control (CAE-06), two plain single-chart autoencoder controls at latent
dimension 20 and 40, and two classical-MDS-coordinate decoder baselines at `p=8` and `p=20`
-- all matched in hidden width (250) and depth (3 layers per side) to the CAE, per CAE-03.

**This notebook trains none of them.** All eight are cached under `notebooks/.cache/` by
plan 02.2-05; this section only asserts every required artifact exists and reports what
training them already cost.


In [3]:
_required_artifacts = (
    [(f"cae_fit_{FIT_KEY}_seed{seed}", "npz") for seed in SEEDS]
    + [(f"cae_fit_meta_{FIT_KEY}_seed{seed}", "json") for seed in SEEDS]
    + [(f"cae_ctrl_relu_{FIT_KEY}", "npz"), (f"cae_ctrl_relu_meta_{FIT_KEY}", "json")]
    + [(f"cae_ctrl_plainae{latent}_{FIT_KEY}", "npz") for latent in PLAIN_AE_LATENTS]
    + [(f"cae_ctrl_plainae{latent}_meta_{FIT_KEY}", "json") for latent in PLAIN_AE_LATENTS]
    + [(f"cae_ctrl_mdsdec{p}_{FIT_KEY}", "npz") for p in MDS_BASELINE_P]
    + [(f"cae_ctrl_mdsdec{p}_meta_{FIT_KEY}", "json") for p in MDS_BASELINE_P]
    + [(f"cae_pairs_{FIT_KEY}", "npz")]
    + [(f"mds_eigenspectrum_{FIT_KEY}", "npz")]
    + [(SUBSAMPLE_STEM, "npz")]
    + [(f"geometry_probes_{FIT_KEY}", "json")]
    + [(f"cae_verdict_{FIT_KEY}", "json")]
)
for _stem, _ext in _required_artifacts:
    _path = cache.cache_path(_stem, _ext)
    assert _path.exists() and _path.stat().st_size > 0, (
        f"Required cache artifact missing or empty: {_path}. This notebook halts rather "
        "than regenerating it -- there is no regeneration path; every fit here is already "
        "trained and cached by plan 02.2-05."
    )
print(f"=== §3: all {len(_required_artifacts)} required cache artifacts present and non-empty ===")

# --- what training them cost ---------------------------------------------------------------


def _stopping_reason(meta: dict) -> str:
    if meta.get("wallclock_truncated"):
        return "wallclock_ceiling"
    if meta.get("early_stopped"):
        return "early_stop_plateau"
    return "epoch_cap"


_meta_specs = (
    [(f"cae_seed_{seed}", f"cae_fit_meta_{FIT_KEY}_seed{seed}") for seed in SEEDS]
    + [("cae_ctrl_relu", f"cae_ctrl_relu_meta_{FIT_KEY}")]
    + [
        (f"cae_ctrl_plainae{latent}", f"cae_ctrl_plainae{latent}_meta_{FIT_KEY}")
        for latent in PLAIN_AE_LATENTS
    ]
    + [(f"cae_ctrl_mdsdec{p}", f"cae_ctrl_mdsdec{p}_meta_{FIT_KEY}") for p in MDS_BASELINE_P]
)

FIT_META = {}
_total_wallclock = 0.0
_hdr = (
    f"{'run_id':>20} {'epochs_run':>11} {'wallclock_s':>12} {'early_stopped':>14} "
    f"{'wallclock_truncated':>20} {'stopping_reason':>18}"
)
print(_hdr)
print("-" * len(_hdr))
for run_id, meta_stem in _meta_specs:
    meta = json.loads(cache.cache_path(meta_stem, "json").read_text())
    FIT_META[run_id] = meta
    reason = _stopping_reason(meta)
    _total_wallclock += meta["wallclock_s"]
    print(
        f"{run_id:>20} {meta['epochs_run']:>11} {meta['wallclock_s']:>12.2f} "
        f"{str(meta['early_stopped']):>14} {str(meta['wallclock_truncated']):>20} {reason:>18}"
    )
print("-" * len(_hdr))
print(
    f"Total wallclock across all eight cached runs: {_total_wallclock:.1f}s "
    f"({_total_wallclock / 3600:.2f}h)"
)

assert all(not m["wallclock_truncated"] for m in FIT_META.values()), (
    "at least one cached run was wallclock-truncated -- this notebook assumes every fit "
    "completed under its own stopping rule, not a wallclock cutoff"
)
print("\nNo cached run was wallclock-truncated.")


=== §3: all 21 required cache artifacts present and non-empty ===
              run_id  epochs_run  wallclock_s  early_stopped  wallclock_truncated    stopping_reason
----------------------------------------------------------------------------------------------------
   cae_seed_20260803          36      1984.85           True                False early_stop_plateau
   cae_seed_20260804          30      1622.51           True                False early_stop_plateau
   cae_seed_20260805          24      1683.54           True                False early_stop_plateau
       cae_ctrl_relu          22      1546.22           True                False early_stop_plateau
  cae_ctrl_plainae20          40        31.71          False                False          epoch_cap
  cae_ctrl_plainae40          40        21.14          False                False          epoch_cap
    cae_ctrl_mdsdec8          40        11.35          False                False          epoch_cap
   cae_ctrl_mdsdec20     

## §4. Reload: rebuild the CAE models, load the fit-artifact arrays, assert split identity

The three CAE seed models are reconstructed as live `nn.Module` instances (via
`cae_mod.arrays_to_state_dict`) because §5-§7 need to run their forward passes. The
ReLU / plain-AE / MDS-decoder controls are **not** rebuilt as models here -- every T3
number this notebook computes (§8) comes from each fit's stored `y_holdout` array, never
from a fresh forward pass, so reconstructing those five models would cost memory and time
for a computation nothing downstream uses. `cae_evaluate_run.py` does rebuild all eight;
this is a deliberate, recorded deviation from that runner (see §11).


In [4]:
_seed_npz = {
    seed: dict(np.load(cache.cache_path(f"cae_fit_{FIT_KEY}_seed{seed}", "npz")))
    for seed in SEEDS
}
_relu_npz = dict(np.load(cache.cache_path(f"cae_ctrl_relu_{FIT_KEY}", "npz")))
_plainae_npz = {
    latent: dict(np.load(cache.cache_path(f"cae_ctrl_plainae{latent}_{FIT_KEY}", "npz")))
    for latent in PLAIN_AE_LATENTS
}
_mdsdec_npz = {
    p: dict(np.load(cache.cache_path(f"cae_ctrl_mdsdec{p}_{FIT_KEY}", "npz")))
    for p in MDS_BASELINE_P
}

# T-02.2-23 mitigation: the persisted train/holdout split index arrays must be bit-identical
# across all eight loaded fits before computing anything -- a metric computed across
# mismatched splits would silently mean nothing.
_all_npz_for_split = (
    list(_seed_npz.values()) + [_relu_npz] + list(_plainae_npz.values()) + list(_mdsdec_npz.values())
)
_canonical_train_idx = _all_npz_for_split[0]["train_idx"]
_canonical_holdout_idx = _all_npz_for_split[0]["holdout_idx"]
for _npz in _all_npz_for_split[1:]:
    assert np.array_equal(_npz["train_idx"], _canonical_train_idx), (
        "train_idx mismatch across the eight cached fit artifacts"
    )
    assert np.array_equal(_npz["holdout_idx"], _canonical_holdout_idx), (
        "holdout_idx mismatch across the eight cached fit artifacts"
    )
train_idx = _canonical_train_idx
holdout_idx = _canonical_holdout_idx
assert train_idx.shape == (8000,), train_idx.shape
assert holdout_idx.shape == (2000,), holdout_idx.shape
print(
    f"split index arrays identical across all eight cached fits: "
    f"train={train_idx.shape[0]} holdout={holdout_idx.shape[0]}"
)

_subsample = dict(np.load(cache.cache_path(SUBSAMPLE_STEM, "npz")))
X = _subsample["legacysurvey"]
assert X.shape == (10_000, 768), X.shape
x_all_t = torch.tensor(X, dtype=torch.float32)
x_train_t = x_all_t[torch.from_numpy(train_idx)]
x_holdout_t = x_all_t[torch.from_numpy(holdout_idx)]


def _build_cae(activation: str) -> "cae_mod.ChartAutoEncoder":
    return cae_mod.ChartAutoEncoder(
        in_dim=768,
        embed_dim=L_EMBED,
        chart_dim=D_CHART,
        n_charts=N_CHARTS_INIT,
        hidden=[HIDDEN_WIDTH, HIDDEN_WIDTH, HIDDEN_WIDTH],
        activation=activation,
    )


def _rebuild_cae(npz: dict, activation: str) -> "cae_mod.ChartAutoEncoder":
    model = _build_cae(activation)
    model.load_state_dict(cae_mod.arrays_to_state_dict(npz, model.state_dict()))
    model.eval()
    return model


# Only the three CAE seed models are rebuilt as live nn.Module instances -- the ReLU,
# plain-AE, and MDS-decoder fits are deliberately NOT reconstructed here (see §11): every
# T3 number comes from their stored y_holdout arrays, not from a forward pass.
seed_models = {seed: _rebuild_cae(_seed_npz[seed], ACTIVATION) for seed in SEEDS}
model_main = seed_models[MAIN_SEED]
print(f"\nrebuilt {len(seed_models)} CAE seed models (main seed = {MAIN_SEED})")

# --- memory discipline: retain only the small downstream arrays, then drop the full npz ----
z_all_main = _seed_npz[MAIN_SEED]["z_all"]
p_all_main = _seed_npz[MAIN_SEED]["p_all"]
y_holdout_by_run = {
    "cae_main": _seed_npz[MAIN_SEED]["y_holdout"],
    "cae_ctrl_relu": _relu_npz["y_holdout"],
    **{f"cae_ctrl_plainae{latent}": _plainae_npz[latent]["y_holdout"] for latent in PLAIN_AE_LATENTS},
    **{f"cae_ctrl_mdsdec{p}": _mdsdec_npz[p]["y_holdout"] for p in MDS_BASELINE_P},
}

del _seed_npz, _relu_npz, _plainae_npz, _mdsdec_npz, _subsample, X, _all_npz_for_split
gc.collect()

_peak_rss_mib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print(
    f"retained arrays: train_idx{train_idx.shape}, holdout_idx{holdout_idx.shape}, "
    f"z_all_main{z_all_main.shape}, p_all_main{p_all_main.shape}, "
    f"{len(y_holdout_by_run)} y_holdout arrays"
)
print(f"peak RSS so far: {_peak_rss_mib:.1f} MiB")


split index arrays identical across all eight cached fits: train=8000 holdout=2000



rebuilt 3 CAE seed models (main seed = 20260803)
retained arrays: train_idx(8000,), holdout_idx(2000,), z_all_main(10000, 40), p_all_main(10000, 16), 6 y_holdout arrays
peak RSS so far: 831.3 MiB


## §5. Chart survival (CAE-05)

Sixteen charts were over-specified on purpose; weight decay was expected to prune the
surplus, and the surviving count is read off *a posteriori* at decoder-weight-norm
tolerance `PRUNE_TOL` -- never chosen as an input. A chart is pruned when its decoder's
weight mass, relative to the largest chart's, falls strictly below `PRUNE_TOL`.


In [5]:
chart_survival_by_seed = {
    seed: cae_mod.chart_survival(seed_models[seed], PRUNE_TOL) for seed in SEEDS
}

print("=== §5: chart survival (CAE-05) ===")
for seed, cs in chart_survival_by_seed.items():
    tag = "  <- main seed" if seed == MAIN_SEED else ""
    print(
        f"seed={seed}: {cs['n_charts_surviving']}/{cs['n_charts_initial']} surviving  "
        f"surviving_indices={cs['surviving_indices']}  pruned_indices={cs['pruned_indices']}{tag}"
    )

chart_survival_main = chart_survival_by_seed[MAIN_SEED]
surviving_indices_main = chart_survival_main["surviving_indices"]

print(f"\nmain seed ({MAIN_SEED}) per-chart mass ratios (prune_tol={PRUNE_TOL}):")
for i, ratio in enumerate(chart_survival_main["chart_mass_ratio"]):
    print(f"  chart {i:>2}: mass_ratio={ratio:.6f}")

assert (
    chart_survival_main["n_charts_surviving"]
    == PUBLISHED["chart_count_surviving"]["n_charts_surviving"]
), "main-seed surviving count does not match the sealed artifact"
assert (
    chart_survival_main["n_charts_initial"] == PUBLISHED["chart_count_surviving"]["n_charts_initial"]
), "main-seed initial count does not match the sealed artifact"
for seed, cs in chart_survival_by_seed.items():
    _pub_cs = PUBLISHED["chart_count_by_seed"][str(seed)]
    assert cs["n_charts_surviving"] == _pub_cs["n_charts_surviving"], (
        f"seed={seed}: surviving={cs['n_charts_surviving']} vs sealed "
        f"{_pub_cs['n_charts_surviving']}"
    )
print("\nAll three seeds' surviving counts reproduce the sealed artifact.")

print(
    "\nAll sixteen initial charts survived at every seed -- weight decay pruned nothing. "
    "That all-sixteen-surviving outcome is itself the finding, not a null result."
)


=== §5: chart survival (CAE-05) ===
seed=20260803: 16/16 surviving  surviving_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  pruned_indices=[]  <- main seed
seed=20260804: 16/16 surviving  surviving_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  pruned_indices=[]
seed=20260805: 16/16 surviving  surviving_indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]  pruned_indices=[]

main seed (20260803) per-chart mass ratios (prune_tol=0.01):
  chart  0: mass_ratio=0.576544
  chart  1: mass_ratio=0.500835
  chart  2: mass_ratio=0.687734
  chart  3: mass_ratio=0.536117
  chart  4: mass_ratio=0.660218
  chart  5: mass_ratio=1.000000
  chart  6: mass_ratio=0.577372
  chart  7: mass_ratio=0.518481
  chart  8: mass_ratio=0.688749
  chart  9: mass_ratio=0.647270
  chart 10: mass_ratio=0.569023
  chart 11: mass_ratio=0.585688
  chart 12: mass_ratio=0.594205
  chart 13: mass_ratio=0.543529
  chart 14: mass_ratio=0.497867
  chart 15: mass_ratio=0.545136

All

## §6. Gate T1 -- geodesic distortion

Measures how far the initial encoder's global embedding `z_all` (never chart-local
coordinates -- two points in different charts share no common parameterisation) distorts
the cached 200,000-pair geodesic sample, after fitting exactly one global isotropic scale
factor on train-only pairs (a neural encoder carries no built-in scale tying it to the
geodesic metric, unlike the classical/Krein coordinates 02.1 measured). Threshold `0.15`,
strict.


In [6]:
_pairs_npz = dict(np.load(cache.cache_path(f"cae_pairs_{FIT_KEY}", "npz")))
pair_rows, pair_cols = _pairs_npz["rows"], _pairs_npz["cols"]

# Read only geo_pairs_r2 out of the mds_eigenspectrum archive, not the whole file (it also
# carries the full 10,000x40 eigenvector block).
with np.load(cache.cache_path(f"mds_eigenspectrum_{FIT_KEY}", "npz")) as _mds_npz:
    geo_pairs_r2 = np.asarray(_mds_npz["geo_pairs_r2"], dtype=np.float64)

train_pair_mask = np.isin(pair_rows, train_idx) & np.isin(pair_cols, train_idx)
holdout_pair_mask = np.isin(pair_rows, holdout_idx) & np.isin(pair_cols, holdout_idx)

t1_all = cae_mod.embedding_distortion(
    z=z_all_main,
    geo_pairs=geo_pairs_r2,
    rows=pair_rows,
    cols=pair_cols,
    train_mask=train_pair_mask,
    chart_dim=D_CHART,
    embed_dim=L_EMBED,
)
T1 = t1_all["median_abs_rel"]

print("=== §6: gate T1 ===")
print(f"median_abs_rel      = {t1_all['median_abs_rel']:.10f}  (gate value)")
print(f"median_signed_rel   = {t1_all['median_signed_rel']:.10f}")
print(f"p95_abs_rel         = {t1_all['p95_abs_rel']:.10f}")
print(f"global_scale_factor = {t1_all['global_scale_factor']:.10f}")
print(f"n_pairs             = {t1_all['n_pairs']}")
print(f"threshold           = {THRESH_DISTORTION_MAX}")
print(f"passed              = {T1 < THRESH_DISTORTION_MAX}")

# Non-gating holdout-only companion: reuse the already-fit global scale factor rather than
# refitting it on the holdout-only subset.
_d2_rep_all = ((z_all_main[pair_rows] - z_all_main[pair_cols]) ** 2).sum(axis=1)
_d2_geo_all = geo_pairs_r2.astype(np.float64) ** 2
t1_holdout_only = gp.distortion_stats(
    t1_all["global_scale_factor"] * _d2_rep_all[holdout_pair_mask], _d2_geo_all[holdout_pair_mask]
)
t1_holdout_only["n_pairs"] = int(holdout_pair_mask.sum())
print(
    f"\n(non-gating) T1 holdout-only ({t1_holdout_only['n_pairs']} pairs): "
    f"median_abs_rel={t1_holdout_only['median_abs_rel']:.10f}"
)


=== §6: gate T1 ===
median_abs_rel      = 0.2969813323  (gate value)
median_signed_rel   = 0.0003670456
p95_abs_rel         = 0.8160568764
global_scale_factor = 0.3467897685
n_pairs             = 200000
threshold           = 0.15
passed              = False

(non-gating) T1 holdout-only (7970 pairs): median_abs_rel=0.2966594415


## §7. Gate T2 -- chart-transition cycle residual (eq. 8)

Measures whether a point encoded once, decoded through chart alpha, re-encoded, and decoded
through chart beta returns close to itself. The residual is normalised by a matched base
quantity -- twice the single-pass argmax-chart reconstruction norm, matching eq. 8's
two-term unsquared form -- so the ratio is scale-free. Evaluated only on holdout rows whose
top two (post-pruning) chart probabilities both clear `OVERLAP_P_MIN`, per
`select_overlap_pairs`. Threshold `2.0`, strict.


In [7]:
p_holdout_main = p_all_main[holdout_idx]  # (n_holdout, N_CHARTS_INIT), order matches x_holdout_t

overlap = cae_mod.select_overlap_pairs(
    p_holdout_main, OVERLAP_P_MIN, OVERLAP_MIN_POINTS, surviving_indices_main
)
overlap_rows, overlap_alpha, overlap_beta = overlap["rows"], overlap["alpha"], overlap["beta"]
print(f"=== §7: gate T2 ===")
print(f"qualifying holdout rows = {len(overlap_rows)}  (>= {OVERLAP_MIN_POINTS} required)")

_pair_groups: dict = {}
for _i in range(len(overlap_rows)):
    _key = (int(overlap_alpha[_i]), int(overlap_beta[_i]))
    _pair_groups.setdefault(_key, []).append(_i)
print(f"distinct (alpha, beta) chart pairs = {len(_pair_groups)}")

r_cycle_vals = np.empty(len(overlap_rows), dtype=np.float64)
base_vals = np.empty(len(overlap_rows), dtype=np.float64)

with torch.no_grad():
    x_overlap = x_holdout_t[torch.from_numpy(overlap_rows)]
    for (_alpha, _beta), _local_positions in _pair_groups.items():
        _idx_t = torch.tensor(_local_positions, dtype=torch.long)
        _xb = x_overlap[_idx_t]

        _r = cae_mod.r_cycle(model_main, _xb, _alpha, _beta)
        r_cycle_vals[_local_positions] = _r.numpy()

        # matched base quantity: 2 * ||x - yhat(x)||, twice the single-pass argmax-chart
        # reconstruction norm, matching eq. 8's two-term unsquared form
        _out = model_main(_xb)
        _chart_amax = _out["p"].argmax(dim=1)
        _y_amax = _out["y_charts"][torch.arange(_xb.shape[0]), _chart_amax]
        _base = 2.0 * torch.linalg.vector_norm(_xb - _y_amax, dim=-1)
        base_vals[_local_positions] = _base.numpy()

mean_r_cycle = float(r_cycle_vals.mean())
mean_base = float(base_vals.mean())
T2 = mean_r_cycle / mean_base

print(f"mean(R_cycle) = {mean_r_cycle:.10f}")
print(f"mean(E_base)  = {mean_base:.10f}")
print(f"ratio (gate value) = {T2:.10f}")
print(f"threshold = {THRESH_RCYCLE_RATIO_MAX}")
print(f"passed = {T2 < THRESH_RCYCLE_RATIO_MAX}")

assert len(overlap_rows) == PUBLISHED["metrics"]["t2"]["n_qualify"], (
    f"qualifying count {len(overlap_rows)} vs sealed {PUBLISHED['metrics']['t2']['n_qualify']}"
)
print("\nQualifying row count reproduces PUBLISHED['metrics']['t2']['n_qualify'].")


=== §7: gate T2 ===
qualifying holdout rows = 2000  (>= 200 required)
distinct (alpha, beta) chart pairs = 83


mean(R_cycle) = 0.6454455853
mean(E_base)  = 0.5924963986
ratio (gate value) = 1.0893662590
threshold = 2.0
passed = True

Qualifying row count reproduces PUBLISHED['metrics']['t2']['n_qualify'].


## §8. Gate T3 -- held-out reconstruction vs. matched-capacity controls

CAE-03's capacity match: same hidden width (250), same three layers per side, identical
training protocol between the CAE and its plain-AE / MDS-decoder controls. The
pre-registration's own rule is `mse_cae < (1 - margin) * mse_control` for **both** gating
controls (`plain_ae20`, `mds_dec8`); dividing through by `mse_control` puts it in the same
strict-less-than value/threshold form T1 and T2 already use, and taking the `max` over the
two ratios encodes the AND -- PASS iff the worse (larger) of the two ratios still clears
`1 - THRESH_RECON_MARGIN`.


In [8]:
def _recon(y_holdout_np: np.ndarray) -> dict:
    y_t = torch.tensor(np.asarray(y_holdout_np, dtype=np.float64), dtype=torch.float64)
    return cae_mod.reconstruction_stats(x_holdout_t.double(), y_t)


recon_cae_main = _recon(y_holdout_by_run["cae_main"])
recon_relu = _recon(y_holdout_by_run["cae_ctrl_relu"])
recon_plainae = {
    latent: _recon(y_holdout_by_run[f"cae_ctrl_plainae{latent}"]) for latent in PLAIN_AE_LATENTS
}
recon_mdsdec = {p: _recon(y_holdout_by_run[f"cae_ctrl_mdsdec{p}"]) for p in MDS_BASELINE_P}

mse_cae = recon_cae_main["mse_per_dim"]
mse_relu = recon_relu["mse_per_dim"]
mse_plainae20 = recon_plainae[20]["mse_per_dim"]
mse_plainae40 = recon_plainae[40]["mse_per_dim"]
mse_mdsdec8 = recon_mdsdec[8]["mse_per_dim"]
mse_mdsdec20 = recon_mdsdec[20]["mse_per_dim"]

ratio_vs_plainae20 = mse_cae / mse_plainae20
ratio_vs_mdsdec8 = mse_cae / mse_mdsdec8
T3 = max(ratio_vs_plainae20, ratio_vs_mdsdec8)
recon_margin_threshold = 1.0 - THRESH_RECON_MARGIN

_DIM_KEYS = ("dim_mse_mean", "dim_mse_median", "dim_mse_p95", "dim_mse_max")
_rows = [
    ("cae_main (SiLU)", recon_cae_main),
    ("relu_control", recon_relu),
    ("plain_ae20", recon_plainae[20]),
    ("plain_ae40", recon_plainae[40]),
    ("mds_dec8", recon_mdsdec[8]),
    ("mds_dec20", recon_mdsdec[20]),
]
print("=== §8: gate T3 -- held-out reconstruction vs. matched-capacity controls ===")
_hdr = (
    f"{'fit':>16} {'mse_per_dim':>14} {'dim_mse_mean':>14} {'dim_mse_median':>16} "
    f"{'dim_mse_p95':>14} {'dim_mse_max':>14}"
)
print(_hdr)
print("-" * len(_hdr))
for name, r in _rows:
    print(
        f"{name:>16} {r['mse_per_dim']:>14.6e} {r['dim_mse_mean']:>14.6e} "
        f"{r['dim_mse_median']:>16.6e} {r['dim_mse_p95']:>14.6e} {r['dim_mse_max']:>14.6e}"
    )
print("-" * len(_hdr))
print(f"\nratio_vs_plain_ae20 = mse_cae/mse_plain_ae20 = {ratio_vs_plainae20:.6f}")
print(f"ratio_vs_mds_dec8   = mse_cae/mse_mds_dec8   = {ratio_vs_mdsdec8:.6f}")
print(f"gate value (max)    = {T3:.6f}")
print(f"threshold           = {recon_margin_threshold:.6f}  (= 1 - THRESH_RECON_MARGIN)")
print(f"passed              = {T3 < recon_margin_threshold}")

_mdsdec8_meta = FIT_META["cae_ctrl_mdsdec8"]
_mdsdec20_meta = FIT_META["cae_ctrl_mdsdec20"]
print(
    f"\n(context, non-gating) linear-decoder MSE floor: "
    f"mds_dec8={_mdsdec8_meta['mse_linear_floor_holdout']:.6e}  "
    f"mds_dec20={_mdsdec20_meta['mse_linear_floor_holdout']:.6e}"
)

# --- regression check: all six fits' mse_per_dim against the sealed artifact ---------------
_pub_t3 = PUBLISHED["metrics"]["t3"]
_pub_act = PUBLISHED["activation_substitution"]
for _name, _val, _exp in (
    ("mse_cae", mse_cae, _pub_t3["mse_cae"]),
    ("mse_plain_ae20", mse_plainae20, _pub_t3["mse_plain_ae20"]),
    ("mse_plain_ae40", mse_plainae40, _pub_t3["mse_plain_ae40"]),
    ("mse_mds_dec8", mse_mdsdec8, _pub_t3["mse_mds_dec8"]),
    ("mse_mds_dec20", mse_mdsdec20, _pub_t3["mse_mds_dec20"]),
    ("mse_relu_control", mse_relu, _pub_act["mse_relu_control"]),
):
    assert abs(_val - _exp) <= 1e-9 * abs(_exp), f"{_name}: {_val!r} vs sealed {_exp!r}"
print("\nAll six fits' mse_per_dim values reproduce the sealed artifact to 1e-9 relative.")


=== §8: gate T3 -- held-out reconstruction vs. matched-capacity controls ===
             fit    mse_per_dim   dim_mse_mean   dim_mse_median    dim_mse_p95    dim_mse_max
---------------------------------------------------------------------------------------------
 cae_main (SiLU)   1.254476e-04   1.254476e-04     1.158926e-04   2.257457e-04   4.460075e-04
    relu_control   1.241643e-04   1.241643e-04     1.143339e-04   2.196367e-04   4.185980e-04
      plain_ae20   3.497920e-05   3.497920e-05     3.369947e-05   5.431005e-05   8.250235e-05
      plain_ae40   3.408036e-05   3.408036e-05     3.296755e-05   5.227945e-05   7.930389e-05
        mds_dec8   5.670749e-05   5.670749e-05     5.419470e-05   9.277743e-05   1.521850e-04
       mds_dec20   5.153384e-05   5.153384e-05     4.954255e-05   8.589216e-05   1.235520e-04
---------------------------------------------------------------------------------------------

ratio_vs_plain_ae20 = mse_cae/mse_plain_ae20 = 3.586350
ratio_vs_mds_dec8   

## §9. Non-gating evidence

Unfaithfulness/coverage (eqs. 20/21), the CAE-06 SiLU-vs-ReLU activation substitution cost,
and 02.1's own classical Krein comparison, reported for context -- none of these three feed
the verdict. "This gate judges the CAE only against its own pre-registered T1/T2/T3
thresholds," as the sealed artifact itself states.


In [9]:
uc = cae_mod.unfaithfulness_coverage(
    model_main, x_train_t, surviving_indices_main, UNFAITHFUL_SAMPLES, UNFAITHFUL_SEED
)
print("=== §9: non-gating evidence ===")
print(f"unfaithfulness = {uc['unfaithfulness']:.10f}")
print(f"coverage       = {uc['coverage']:.10f}")
print(f"n_samples      = {uc['n_samples']}")
print(f"n_distinct     = {uc['n_distinct']}")

assert abs(uc["unfaithfulness"] - PUBLISHED["unfaithfulness"]) <= 1e-6 * abs(
    PUBLISHED["unfaithfulness"]
), f"unfaithfulness {uc['unfaithfulness']!r} vs sealed {PUBLISHED['unfaithfulness']!r}"
assert abs(uc["coverage"] - PUBLISHED["coverage"]) <= 1e-6 * max(abs(PUBLISHED["coverage"]), 1e-12), (
    f"coverage {uc['coverage']!r} vs sealed {PUBLISHED['coverage']!r}"
)
print("unfaithfulness and coverage reproduce the sealed artifact to 1e-6 relative.")

# unfaithfulness_coverage's internal pairwise distance allocates roughly
# n_samples x n_train (1000 x 8000) -- free its intermediates once the two scalars are read.
del uc
gc.collect()

# --- CAE-06 activation substitution ---------------------------------------------------------
delta_relu_minus_silu = mse_relu - mse_cae
print(f"\nCAE-06 activation substitution: mse_relu_control={mse_relu:.6e}  mse_silu_main={mse_cae:.6e}")
print(
    f"delta(relu - silu) = {delta_relu_minus_silu:.6e}  "
    "(positive means the pre-registered SiLU activation reconstructs BETTER than the "
    "ReLU control -- lower held-out MSE, a benefit not a cost; negative would mean the "
    "SiLU substitution costs reconstruction quality relative to ReLU)"
)

# --- 02.1's classical q=0 Krein comparison, reported context only --------------------------
_geometry_probes = json.loads(cache.cache_path(f"geometry_probes_{FIT_KEY}", "json").read_text())
_krein_q0 = {row["p"]: row for row in _geometry_probes["krein_ladder"] if row["q"] == 0}
print("\n(reported, non-gating context from Phase 02.1) classical q=0 Krein comparison:")
print(f"  p=40: median_abs_rel = {_krein_q0[40]['median_abs_rel']:.6f}")
print(f"  p=8:  median_abs_rel = {_krein_q0[8]['median_abs_rel']:.6f}")
print(f"  working_dimension = {_geometry_probes['working_dimension']}")
print(
    "  This gate judges the CAE only against its own pre-registered T1/T2/T3 thresholds; "
    "the Krein numbers above are reported for context, never gating."
)


=== §9: non-gating evidence ===
unfaithfulness = 0.0423333608
coverage       = 0.0240000000
n_samples      = 1000
n_distinct     = 24
unfaithfulness and coverage reproduce the sealed artifact to 1e-6 relative.



CAE-06 activation substitution: mse_relu_control=1.241643e-04  mse_silu_main=1.254476e-04
delta(relu - silu) = -1.283300e-06  (positive means the pre-registered SiLU activation reconstructs BETTER than the ReLU control -- lower held-out MSE, a benefit not a cost; negative would mean the SiLU substitution costs reconstruction quality relative to ReLU)

(reported, non-gating context from Phase 02.1) classical q=0 Krein comparison:
  p=40: median_abs_rel = 0.179641
  p=8:  median_abs_rel = 0.121024
  working_dimension = {'classical_p': 8, 'criterion': "Maximum-curvature (kneedle) elbow on the Tenenbaum residual-variance curve (1 - R^2 between geodesic and embedded pairwise distances): both axes are normalized to [0, 1] by their own range, and the elbow is the point of greatest perpendicular distance from the chord connecting the curve's first and last normalized point. Swept over d = 1..40 (K_EFF, bounded by D_SWEEP_MAX=40 and N_POSITIVE=4971); ties broken to the lower d (ELBOW_TIE_BREAK

## §10. Verdict -- applying the pre-registered rule

The rule (Section 5) is a conjunction of three strict-less-than comparisons with no
MARGINAL tier, applied mechanically to the three gate values computed in §6-§8. This
section assembles them, calls the tested `cae_mod.verdict_from_metrics` (never
re-derived inline), and then regression-checks every gate value and the verdict string
itself against the sealed artifact -- the entire evidence that this notebook's distillation
is faithful, not just a plausible-looking retelling.


In [10]:
metrics = {"distortion": T1, "rcycle_ratio": T2, "recon_margin": T3}
thresholds = {
    "distortion": THRESH_DISTORTION_MAX,
    "rcycle_ratio": THRESH_RCYCLE_RATIO_MAX,
    "recon_margin": recon_margin_threshold,
}
assert set(metrics) == set(cae_mod.GATING_METRICS), (
    f"metrics keys {set(metrics)} do not match cae_mod.GATING_METRICS {set(cae_mod.GATING_METRICS)}"
)

verdict, gate_detail = cae_mod.verdict_from_metrics(metrics, thresholds)

_gate_desc = {
    "distortion": "T1: geodesic distortion of global embedding",
    "rcycle_ratio": "T2: chart-transition cycle residual ratio",
    "recon_margin": "T3: held-out reconstruction margin (max ratio)",
}
_hdr = f"{'gate':>14} {'what it measures':>46} {'value':>14} {'threshold':>12} {'passed':>8}"
print("=" * len(_hdr))
print("CAE VERDICT TABLE  (conjunction of three strict-less-than gates, no MARGINAL tier)")
print("=" * len(_hdr))
print(_hdr)
print("-" * len(_hdr))
for gate in cae_mod.GATING_METRICS:
    d = gate_detail[gate]
    print(
        f"{gate:>14} {_gate_desc[gate]:>46} {d['value']:>14.6f} {d['threshold']:>12.6f} "
        f"{str(d['passed']):>8}"
    )
print("-" * len(_hdr))
print(f"\nCAE_VERDICT = {verdict}")

# --- full regression block against the sealed artifact -------------------------------------
print("\n=== §10: regression check against cae_verdict_43cf438bc944c509.json ===")
assert abs(metrics["distortion"] - PUBLISHED["metrics"]["distortion"]) <= 1e-9 * abs(
    PUBLISHED["metrics"]["distortion"]
), f"distortion {metrics['distortion']!r} vs sealed {PUBLISHED['metrics']['distortion']!r}"
assert abs(metrics["recon_margin"] - PUBLISHED["metrics"]["recon_margin"]) <= 1e-9 * abs(
    PUBLISHED["metrics"]["recon_margin"]
), f"recon_margin {metrics['recon_margin']!r} vs sealed {PUBLISHED['metrics']['recon_margin']!r}"
# rcycle_ratio routes through model forward passes (not pure array algebra), so it is
# regression-checked at the looser 1e-6 relative tolerance rather than 1e-9.
assert abs(metrics["rcycle_ratio"] - PUBLISHED["metrics"]["rcycle_ratio"]) <= 1e-6 * abs(
    PUBLISHED["metrics"]["rcycle_ratio"]
), f"rcycle_ratio {metrics['rcycle_ratio']!r} vs sealed {PUBLISHED['metrics']['rcycle_ratio']!r}"

print(f"distortion:    notebook={metrics['distortion']:.17g}  sealed={PUBLISHED['metrics']['distortion']:.17g}")
print(f"rcycle_ratio:  notebook={metrics['rcycle_ratio']:.17g}  sealed={PUBLISHED['metrics']['rcycle_ratio']:.17g}")
print(f"recon_margin:  notebook={metrics['recon_margin']:.17g}  sealed={PUBLISHED['metrics']['recon_margin']:.17g}")

assert verdict == PUBLISHED["CAE_VERDICT"], f"verdict {verdict} vs sealed {PUBLISHED['CAE_VERDICT']}"

_pub_gate_passed = {
    "distortion": PUBLISHED["metrics"]["distortion"] < PUBLISHED["thresholds"]["distortion"],
    "rcycle_ratio": PUBLISHED["metrics"]["rcycle_ratio"] < PUBLISHED["thresholds"]["rcycle_ratio"],
    "recon_margin": PUBLISHED["metrics"]["recon_margin"] < PUBLISHED["thresholds"]["recon_margin"],
}
for gate in cae_mod.GATING_METRICS:
    assert gate_detail[gate]["passed"] == _pub_gate_passed[gate], (
        f"gate {gate} passed={gate_detail[gate]['passed']} does not match the sign of the "
        f"sealed value against its sealed threshold ({_pub_gate_passed[gate]})"
    )

print(
    f"\nAll three gate values reproduce the sealed artifact within tolerance, the recomputed "
    f"verdict equals the sealed verdict ({PUBLISHED['CAE_VERDICT']}), and each gate's passed "
    "boolean matches the sign of its sealed value against its sealed threshold."
)

print(
    f"\nREGRESSION_OK distortion={metrics['distortion']:.17g} "
    f"rcycle_ratio={metrics['rcycle_ratio']:.17g} "
    f"recon_margin={metrics['recon_margin']:.17g} verdict={verdict}"
)

print(
    "\nThis FAIL is a measured outcome reproduced from cached weights, not a result being "
    "retried until it passes."
)


CAE VERDICT TABLE  (conjunction of three strict-less-than gates, no MARGINAL tier)
          gate                               what it measures          value    threshold   passed
--------------------------------------------------------------------------------------------------
    distortion    T1: geodesic distortion of global embedding       0.296981     0.150000    False
  rcycle_ratio      T2: chart-transition cycle residual ratio       1.089366     2.000000     True
  recon_margin T3: held-out reconstruction margin (max ratio)       3.586350     0.900000    False
--------------------------------------------------------------------------------------------------

CAE_VERDICT = FAIL

=== §10: regression check against cae_verdict_43cf438bc944c509.json ===
distortion:    notebook=0.29698133226319146  sealed=0.29698133226319146
rcycle_ratio:  notebook=1.0893662590388085  sealed=1.0893662590388085
recon_margin:  notebook=3.5863496159842887  sealed=3.5863496159842887

All three gate va

## §11. What this notebook does not reproduce, and why

**Reproduced here.** The three gate metrics (T1 distortion, T2 chart-transition cycle
residual ratio, T3 held-out reconstruction margin), the verdict they resolve to, chart
survival across all three seeds, unfaithfulness and coverage, the per-fit reconstruction
table, and the training cost of all eight cached fits -- every one of them regression-
checked in §5, §7, §8, §9 and §10 against `cae_verdict_43cf438bc944c509.json`.

**Not reproduced here, with the reason and whether it moves a gate value.**

1. **Pre-registration git-ancestry proof** (`cae_train_run.py` STEP 0b and
   `cae_evaluate_run.py` STEP 0b, duplicated verbatim across both files): each shells out to
   `git log --diff-filter=A --format=%H -- <PREREG_PATH>` to find the commit that added
   `02.2-PREREGISTRATION.md`, then `git merge-base --is-ancestor <that-sha> HEAD` to prove
   it precedes HEAD (CAE-01). **Does not move a gate value** -- it is a property of the
   repository's commit history, established once, before the sealed run, and does not
   change under re-execution. See the closing note below on why this notebook cites it
   rather than re-running it.

2. **Geodesic pair-sample redraw self-check** (`cae_train_run.py` STEP 0c): redraws the
   200,000-pair sample under `PAIR_SEED` and, unable to compare it against the un-loaded
   1.55 GiB Isomap distance-matrix pickle, asserts only that a second, differently-seeded
   redraw (`geo_pairs_r2_check`) differs from the first -- confirming the draw is
   seed-dependent, not that either draw matches the pickle. **Does not move a gate value**
   -- this notebook reads the already-persisted `cae_pairs_43cf438bc944c509.npz` rows/cols
   directly (§6), the same array both runners ultimately used.

3. **`_protocol_cfg`** (`cae_train_run.py`): reflects roughly eighteen training-protocol
   constants (architecture, optimiser, wallclock ceiling, split) into every cache key, so
   any protocol edit invalidates every cached fit rather than silently reusing a stale one.
   **Does not move a gate value** -- it is a cache-invalidation discipline, not a
   computation this notebook's regression check depends on.

4. **`run_and_cache`'s `box` closure** (`cae_train_run.py`): threads a mutable dict so a
   single `train_fn()` call feeds both the npz array cache and the json meta cache without
   training twice. **Does not move a gate value** -- pure plumbing around a training call
   this notebook never makes.

5. **The fifty-step timing probe** (`cae_train_run.py` STEP 2, `cae_mod.timing_probe`):
   runs fifty real training steps on a throwaway model to project whether a full 40-epoch
   fit would exceed `WALLCLOCK_CEILING_S=7200`, and if so flips `lip_every_n_steps` from 1
   to 4 for every fit in the protocol. **Does not move a gate value for these eight cached
   fits** -- the probe's own recorded branch was "within ceiling" (§3's wallclock table
   shows none of the eight runs wallclock-truncated), so `lip_every_n_steps` stayed at its
   pre-registered value of 1 throughout.

6. **`cae_evaluate_run.py` importing `cae_train_run`**: the evaluation runner imports
   roughly twenty-five module-level constants (activation, dims, seeds, thresholds, path
   stems, `build_cae`/`build_plain_ae`/`build_mlp_decoder`) from the training runner, which
   means importing it re-executes that runner's own preconditions (STEP 0a-0c above) and
   its entire fit registry as a cache-hit read. **Does not move a gate value** -- a
   cache-hit read returns the identical cached arrays this notebook also loads directly in
   §4, just via an indirection this notebook skips.

7. **Five models rebuilt in `cae_evaluate_run.py` that no metric consumes**
   (`relu_model`, `plainae_models[20]`, `plainae_models[40]`, `mdsdec_models[8]`,
   `mdsdec_models[20]`): STEP 1 there reconstructs live `nn.Module` instances for all five
   controls, but every T3 number comes from each fit's stored `y_holdout` array (STEP 5
   there, §8 here), never from a forward pass through these five. **Does not move a gate
   value** -- §4 above records this same deviation explicitly: this notebook rebuilds only
   the three CAE seed models and reads `y_holdout` straight from the cached npz for every
   control.

8. **Prose fields on the verdict artifact** (`assumptions`, `remediation`,
   `sign_convention`, `recon_margin_gate_note` in `cae_verdict_43cf438bc944c509.json`): five
   assumption statements, three remediation options, and two explanatory paragraphs written
   by `cae_evaluate_run.py` alongside the numeric verdict. **Does not move a gate value** --
   they are prose context around the three numbers this notebook independently recomputes
   and checks in §10; §8 and §9 above restate the load-bearing pieces (the recon-margin
   algebra, the activation-substitution sign convention) inline instead.

9. **Training itself** -- eight fits (three CAE seeds, one ReLU control, two plain-AE
   controls, two MDS-decoder baselines), each running `cae_mod.train_cae` or a sibling
   trainer for up to 40 epochs, collectively costing roughly the wallclock total §3 reports.
   **Is** what produced the gate values, by construction -- but this notebook never re-runs
   it: every model here is `arrays_to_state_dict`-loaded from the npz artifacts plan 02.2-05
   already produced, exactly the reload-not-retrain contract stated in the framing cell.

CAE-01's pre-registration ordering (item 1) is a property of the repository's commit
history that was established before the sealed run and cannot be re-established by
re-running a check now -- a fresh `git merge-base --is-ancestor` today would prove only that
the commit still precedes today's HEAD, not that it preceded the run that actually produced
`cae_verdict_43cf438bc944c509.json` on 2026-08-04. That is precisely why this notebook cites
the ordering (§0/§1) rather than re-executing STEP 0b.

The numbers in §6-§10 came out of the cached weights unchanged, so everything in the list
above is scaffolding the result did not depend on -- the reader can weigh that against the
cost of maintaining 1,280 lines of runner script to produce three numbers and a verdict.
